# model training

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix, load_npz

In [2]:
ROOT = Path().resolve().parent
MATRIX_FILE = ROOT / "data" / "processed" / "movielens_matrix.npz"
USER_MAP_FILE = ROOT / "data" / "processed" / "user_mapping.json"
ITEM_MAP_FILE = ROOT / "data" / "processed" / "item_mapping.json"
MODEL_FILE    = ROOT / "models" / "als_model.pkl"
MAPPINGS_FILE = ROOT / "models" / "mappings.json"

In [3]:
matrix = load_npz(str(MATRIX_FILE))
user_map = json.loads(USER_MAP_FILE.read_text())
item_map = json.loads(ITEM_MAP_FILE.read_text())

In [4]:
rng = np.random.default_rng(42)
matrix = matrix.tocsr()
train = matrix.copy().tolil()
test  = matrix.copy().tolil()

for user in range(matrix.shape[0]):
    row = matrix.getrow(user).indices
    if len(row) < 2:
        test[user, :] = 0
        continue
    n_test = max(1, int(len(row) * 0.2))
    test_items = rng.choice(row, size=n_test, replace=False)
    train_items = np.setdiff1d(row, test_items)
    # zero out the complementary side
    for col in test_items:
        train[user, col] = 0
    for col in train_items:
        test[user, col] = 0

train_csr = train.tocsr()
test_csr  = test.tocsr()
train_csr.eliminate_zeros()
test_csr.eliminate_zeros()

In [5]:
conf = train_csr.copy()
conf.data = 1.0 + 40.0 * conf.data

In [6]:
model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20, random_state=42)

z:\Personal project\recommendation_system\.env\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [7]:
model.fit(conf.tocsr())

  0%|          | 0/20 [00:00<?, ?it/s]

In [8]:
rng = np.random.default_rng(0)
n_users = train_csr.shape[0]
candidates = np.where(np.diff(test_csr.indptr) > 0)[0]  # users with test items
sample = rng.choice(candidates, size=min(500, len(candidates)), replace=False)

precisions, recalls = [], []
# user_items passed to recommend must be the user-item training row (items already seen)
user_items = train_csr.tocsr()

for user in sample:
    test_items = set(test_csr.getrow(user).indices)
    if not test_items:
        continue
    recs = model.recommend(
        user,
        user_items[user],
        N=10,
        filter_already_liked_items=True,
    )
    rec_ids = set(recs[0].tolist())
    hits = len(rec_ids & test_items)
    precisions.append(hits / 10)
    recalls.append(hits / len(test_items))

precision = float(np.mean(precisions))
recall    = float(np.mean(recalls))

In [9]:
print("precision_at_10", precision)
print("recall_at_10", recall)

precision_at_10 0.18580000000000002
recall_at_10 0.08516776877368397


In [10]:
MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(MODEL_FILE, "wb") as f:
    pickle.dump(model, f)
size_kb = MODEL_FILE.stat().st_size / 1024
print(f"Model saved to {MODEL_FILE} ({size_kb:.2f} KB)")

Model saved to Z:\Personal project\recommendation_system\models\als_model.pkl (439.78 KB)


In [11]:
MAPPINGS_FILE.parent.mkdir(parents=True, exist_ok=True)
payload = {"user_to_idx": user_map, "item_to_idx": item_map}
MAPPINGS_FILE.write_text(json.dumps(payload, indent=2))

36996

# optimisation

## avec Cross Validation

In [31]:
def train_test_split(
    matrix: csr_matrix, test_fraction: float, seed: int = 42
) -> tuple[csr_matrix, csr_matrix]:
    """Hold out `test_fraction` of non-zero entries per user as a test mask."""
    rng = np.random.default_rng(seed)
    matrix = matrix.tocsr()
    train = matrix.copy().tolil()
    test  = matrix.copy().tolil()

    for user in range(matrix.shape[0]):
        row = matrix.getrow(user).indices
        if len(row) < 2:
            test[user, :] = 0
            continue
        n_test = max(1, int(len(row) * test_fraction))
        test_items = rng.choice(row, size=n_test, replace=False)
        train_items = np.setdiff1d(row, test_items)
        for col in test_items:
            train[user, col] = 0
        for col in train_items:
            test[user, col] = 0

    train_csr = train.tocsr()
    test_csr  = test.tocsr()
    train_csr.eliminate_zeros()
    test_csr.eliminate_zeros()
    return train_csr, test_csr

def build_confidence_matrix(matrix: csr_matrix, alpha: float) -> csr_matrix:
    """Convert explicit ratings to implicit confidence: c_ui = 1 + alpha * r_ui."""
    conf = matrix.copy().astype(np.float32)
    conf.data = 1.0 + alpha * conf.data
    return conf

def precision_recall_at_k(
    model: AlternatingLeastSquares,
    train_matrix: csr_matrix,
    test_matrix: csr_matrix,
    k: int,
    max_users: int = 500,
) -> tuple[float, float]:
    """Compute mean precision@k and recall@k over a sample of users."""
    rng = np.random.default_rng(0)
    candidates = np.where(np.diff(test_matrix.indptr) > 0)[0]
    sample = rng.choice(candidates, size=min(max_users, len(candidates)), replace=False)

    precisions, recalls = [], []
    user_items = train_matrix.tocsr()

    for user in sample:
        test_items = set(test_matrix.getrow(user).indices)
        if not test_items:
            continue
        recs = model.recommend(user, user_items[user], N=k, filter_already_liked_items=True)
        rec_ids = set(recs[0].tolist())
        hits = len(rec_ids & test_items)
        precisions.append(hits / k)
        recalls.append(hits / len(test_items))

    precision = float(np.mean(precisions))
    recall    = float(np.mean(recalls))
    return precision, recall

In [38]:
precision = []
recall = []
f1 = []

for split in range(5):
    print(f"Split {split + 1}/5")
    train_csr, test_csr = train_test_split(matrix, test_fraction=0.2, seed=42 + split)
    conf = build_confidence_matrix(train_csr, alpha=40.0)
    model.fit(conf.tocsr())
    precision_score, recall_score = precision_recall_at_k(model, train_csr, test_csr, k=10)
    f1_score = 2 * (precision_score * recall_score) / (precision_score + recall_score) if (precision_score + recall_score) > 0 else 0
    precision.append(precision_score)
    recall.append(recall_score)
    f1.append(f1_score)
    print(f"precision@10: {precision_score:.4f}, recall@10: {recall_score:.4f}, f1@10: {f1_score:.4f}\n")

Split 1/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.1826, recall@10: 0.0854, f1@10: 0.1164

Split 2/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.1814, recall@10: 0.0881, f1@10: 0.1186

Split 3/5


...

precision@10: 0.1828, recall@10: 0.0874, f1@10: 0.1183

Split 4/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.1810, recall@10: 0.0859, f1@10: 0.1165

Split 5/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.1812, recall@10: 0.0890, f1@10: 0.1194



In [40]:
print(f"Average precision@10 avec cv: {np.mean(precision):.4f}")
print(f"Average recall@10 avec cv: {np.mean(recall):.4f}")
print(f"Average f1@10 avec cv: {np.mean(f1):.4f}")

Average precision@10 avec cv: 0.1818
Average recall@10 avec cv: 0.0872
Average f1@10 avec cv: 0.1178


## optuna

In [41]:
import optuna

def evaluate_model(model, train_csr, test_csr, sample_size=500):
    rng = np.random.default_rng(0)
    candidates = np.where(np.diff(test_csr.indptr) > 0)[0]
    sample = rng.choice(candidates, size=min(sample_size, len(candidates)), replace=False)

    precisions, recalls = [], []
    user_items = train_csr.tocsr()

    for user in sample:
        test_items = set(test_csr.getrow(user).indices)
        if not test_items:
            continue
        recs = model.recommend(
            user,
            user_items[user],
            N=10,
            filter_already_liked_items=True,
        )
        rec_ids = set(recs[0].tolist())
        hits = len(rec_ids & test_items)
        precisions.append(hits / 10)
        recalls.append(hits / len(test_items))

    return float(np.mean(precisions)), float(np.mean(recalls))

def objective(trial):
    model = AlternatingLeastSquares(
        factors=trial.suggest_int("factors", 20, 200, step=50),
        regularization=trial.suggest_float("regularization", 0.001, 1.0, log=True),
        iterations=trial.suggest_int("iterations", 10, 100, step=10),
        random_state=42,
    )
    model.fit(conf.tocsr())

    precision, recall = evaluate_model(model, train_csr, test_csr)

    # optimise le F1 (équilibre précision/rappel)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

# --- Résultats ---
print("Meilleurs paramètres :", study.best_params)
print(f"Meilleur F1          : {study.best_value:.4f}")

# Réentraîner avec les meilleurs params
best_model = AlternatingLeastSquares(**study.best_params, random_state=42)
best_model.fit(conf.tocsr())

precision, recall = evaluate_model(best_model, train_csr, test_csr)

[I 2026-05-17 16:36:37,030] A new study created in memory with name: no-name-9fc451ab-a4e3-43a3-b1c3-9b0e15e40060


  0%|          | 0/30 [00:00<?, ?it/s]

z:\Personal project\recommendation_system\.env\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 200] and step=50, but the range is not divisible by `step`. It will be replaced with [20, 170].
  optuna_warn(


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-05-17 16:36:37,399] Trial 0 finished with value: 0.13484745109613114 and parameters: {'factors': 70, 'regularization': 0.007187982460268487, 'iterations': 60}. Best is trial 0 with value: 0.13484745109613114.


...

  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-05-17 16:36:46,328] Trial 28 finished with value: 0.18562100798133535 and parameters: {'factors': 170, 'regularization': 0.2253907299362735, 'iterations': 20}. Best is trial 28 with value: 0.18562100798133535.


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-05-17 16:36:46,609] Trial 29 finished with value: 0.18647616870678257 and parameters: {'factors': 170, 'regularization': 0.24289543990343948, 'iterations': 20}. Best is trial 29 with value: 0.18647616870678257.
Meilleurs paramètres : {'factors': 170, 'regularization': 0.24289543990343948, 'iterations': 20}
Meilleur F1          : 0.1865


  0%|          | 0/20 [00:00<?, ?it/s]

In [42]:
print(f"Precision@10 : {precision:.4f}")
print(f"Recall@10    : {recall:.4f}")

Precision@10 : 0.3308
Recall@10    : 0.1298


In [20]:
BEST = ROOT / "models" / "best_model.pkl"

with open(BEST, "wb") as f:
    pickle.dump(best_model, f)
size_kb = BEST.stat().st_size / 1024
print(f"Model saved to {BEST} ({size_kb:.2f} KB)")

Model saved to Z:\Personal project\recommendation_system\models\best_model.pkl (1494.00 KB)


In [21]:
path_params = ROOT / "models" / "best_params.json"
with open(path_params, "w") as f:
    json.dump(study.best_params, f, indent=2)

## optuna avec cv

In [43]:
precision = []
recall = []
f1 = []

for split in range(5):
    print(f"Split {split + 1}/5")
    train_csr, test_csr = train_test_split(matrix, test_fraction=0.2, seed=42 + split)
    conf = build_confidence_matrix(train_csr, alpha=40.0)
    best_model.fit(conf.tocsr())
    precision_score, recall_score = precision_recall_at_k(best_model, train_csr, test_csr, k=10)
    f1_score = 2 * (precision_score * recall_score) / (precision_score + recall_score) if (precision_score + recall_score) > 0 else 0
    precision.append(precision_score)
    recall.append(recall_score)
    f1.append(f1_score)
    print(f"precision@10: {precision_score:.4f}, recall@10: {recall_score:.4f}, f1@10: {f1_score:.4f}\n")

Split 1/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.4784, recall@10: 0.1940, f1@10: 0.2760

Split 2/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.5060, recall@10: 0.2068, f1@10: 0.2936

Split 3/5


...

precision@10: 0.5334, recall@10: 0.2177, f1@10: 0.3092

Split 4/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.5506, recall@10: 0.2238, f1@10: 0.3183

Split 5/5


  0%|          | 0/20 [00:00<?, ?it/s]

precision@10: 0.5298, recall@10: 0.2152, f1@10: 0.3060



In [44]:
print(f"Average precision@10 avec cv: {np.mean(precision):.4f}")
print(f"Average recall@10 avec cv: {np.mean(recall):.4f}")
print(f"Average f1@10 avec cv: {np.mean(f1):.4f}")

Average precision@10 avec cv: 0.5196
Average recall@10 avec cv: 0.2115
Average f1@10 avec cv: 0.3006


# Prediction

In [33]:
from dataclasses import dataclass, field

In [34]:
BASE = Path().resolve().parent
MODEL_FILE    = BASE / "models" / "als_model.pkl"
MAPPINGS_FILE = BASE / "models" / "mappings.json"
MATRIX_FILE   = BASE / "data" / "processed" / "movielens_matrix.npz"
ITEMS_FILE    = BASE / "data" / "raw" / "movielens" / "u.item"

In [35]:
class ModelNotLoadedError(RuntimeError):
    """Raised when inference is attempted before the model artefacts are ready."""


class UnknownUserError(ValueError):
    """Raised when a user_id has no entry in the training mappings."""

In [36]:
@dataclass
class Recommendation:
    item_id: int
    title: str
    score: float

    def to_dict(self) -> dict:
        return {"item_id": self.item_id, "title": self.title, "score": round(self.score, 6)}
    
@dataclass
class _ModelStore:
    model: AlternatingLeastSquares | None = field(default=None, repr=False)
    user_to_idx: dict[str, int] = field(default_factory=dict)
    item_to_idx: dict[str, int] = field(default_factory=dict)
    idx_to_item: dict[int, int] = field(default_factory=dict)   # 0-based idx → original item_id
    item_titles: dict[int, str] = field(default_factory=dict)   # original item_id → title
    user_items: csr_matrix | None = field(default=None, repr=False)

    @property
    def is_loaded(self) -> bool:
        return self.model is not None

    def load(self) -> None:
        for path in (MODEL_FILE, MAPPINGS_FILE, MATRIX_FILE):
            if not path.exists():
                raise ModelNotLoadedError(
                    f"Artefact not found: {path} — run src/recommender/train.py first"
                )
        with open(MODEL_FILE, "rb") as f:
            self.model = pickle.load(f)

        payload = json.loads(MAPPINGS_FILE.read_text())
        self.user_to_idx = payload["user_to_idx"]
        self.item_to_idx = payload["item_to_idx"]
        self.idx_to_item = {idx: int(iid) for iid, idx in self.item_to_idx.items()}

        self.user_items = load_npz(str(MATRIX_FILE)).astype(np.float32).tocsr()

        self.item_titles = _load_item_titles(ITEMS_FILE)

_store = _ModelStore()

def _load_item_titles(path: Path) -> dict[int, str]:
    """Parse u.item (pipe-separated, latin-1) and return {item_id: title}."""
    if not path.exists():
        return {}
    titles: dict[int, str] = {}
    with open(path, encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) >= 2:
                try:
                    titles[int(parts[0])] = parts[1]
                except ValueError:
                    continue
    return titles


def _ensure_loaded() -> None:
    if not _store.is_loaded:
        _store.load()

TypeError: unsupported operand type(s) for |: 'function' and 'NoneType'